# Week 2 — IndoBERT sentiment experiment

This notebook prepares the first real sentiment experiment for SVARA AI. It loads the pinned SmSA source, preserves the official train/validation/test boundaries, applies preprocessing-v1, and removes normalized text overlap from the training view.

The notebook is opt-in for model training. RUN_TRAINING = False keeps a normal notebook or CI run lightweight. Test evaluation has a separate freeze gate. IGAR is an external/domain validation sample only; it is not used for tuning, checkpoint selection, or confirmation-set decisions.

In [ ]:
from __future__ import annotations

import csv
import json
import sys
from collections import Counter
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'data').exists() else NOTEBOOK_DIR.parent
if not (ROOT / 'data').exists():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks/ directory')

sys.path.insert(0, str(ROOT / 'backend'))
from app.ai.preprocessing import (  # noqa: E402
    PREPROCESSING_VERSION,
    PreparedRow,
    normalize_text,
    prepare_labeled_rows,
)

SMSA_DIR = ROOT / 'data/raw/smsa'
IGAR_SAMPLE = ROOT / 'data/samples/igar/Rating_labeled_sample.csv'
ARTIFACT_DIR = ROOT / 'artifacts/week2'
SPLIT_FILES = {
    'train': 'train_preprocess.tsv',
    'validation': 'valid_preprocess.tsv',
    'test': 'test_preprocess.tsv',
}
LABELS = ('positive', 'neutral', 'negative')
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}
print(f'preprocessing={PREPROCESSING_VERSION}; root={ROOT}')

## 1. Load and prepare SmSA

SmSA files are headerless TSV files with sentence and label. The helper keeps punctuation, emoji, slang, and negation cues intact. Same-label duplicates are removed deterministically; missing text is skipped; conflicting labels fail closed.

In [ ]:
def read_smsa(path: Path) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        for line_number, values in enumerate(csv.reader(handle, delimiter='\t'), start=1):
            if not values or all(not value.strip() for value in values):
                continue
            if len(values) != 2:
                raise ValueError(f'{path}: line {line_number} is not sentence<TAB>label')
            rows.append({'sentence': values[0], 'label': values[1]})
    return rows

raw_splits: dict[str, list[dict[str, str]]] = {}
prepared_splits: dict[str, list[PreparedRow]] = {}
reports = {}
for split, filename in SPLIT_FILES.items():
    raw_rows = read_smsa(SMSA_DIR / filename)
    prepared, report = prepare_labeled_rows(
        raw_rows, text_column='sentence', label_column='label'
    )
    raw_splits[split] = raw_rows
    prepared_splits[split] = prepared
    reports[split] = report

validation_test_texts = {
    normalize_text(row.text).casefold()
    for split in ('validation', 'test')
    for row in prepared_splits[split]
}
train_before = prepared_splits['train']
prepared_splits['train'] = [
    row
    for row in train_before
    if normalize_text(row.text).casefold() not in validation_test_texts
]
removed_cross_split = len(train_before) - len(prepared_splits['train'])

for split, rows in prepared_splits.items():
    print(
        split,
        'rows=', len(rows),
        'labels=', dict(Counter(row.label for row in rows)),
        'source_rows=', (rows[0].source_row_number, rows[-1].source_row_number),
    )
print('removed cross-split training rows=', removed_cross_split)

## 2. EDA and label distribution

The tracked JSON and CSV outputs under artifacts/week2/ are generated by scripts/prepare_week2_data.py. This cell provides a dependency-light table view.

In [ ]:
eda = json.loads((ARTIFACT_DIR / 'smsa_eda.json').read_text(encoding='utf-8'))
for split, summary in eda['leakage_safe_splits'].items():
    counts = {
        label: values['count']
        for label, values in summary['label_distribution'].items()
    }
    print(split, 'rows=', summary['usable_rows'], 'labels=', counts)
print('raw cross-split overlap:', eda['cross_split_duplicate_text'])
print('leakage-safe overlap:', eda['leakage_safe_cross_split_duplicate_text'])
print('leakage check passed:', eda['leakage_check_passed'])

## 3. Experiment configuration

indobenchmark/indobert-base-p1 is the initial checkpoint candidate. The revision and hyperparameters are intentionally not frozen until Week 3 evaluation. The model is fine-tuned only on SmSA train and validation; the official test split is evaluated once after selection.

In [ ]:
MODEL_NAME = 'indobenchmark/indobert-base-p1'
MODEL_REVISION = 'main'  # pin a model commit when the Week 3 baseline is frozen
MAX_LENGTH = 128
RUN_TRAINING = False
RUN_TEST_EVALUATION = False
MODEL_FROZEN = False
SEED = 42

if RUN_TEST_EVALUATION and not MODEL_FROZEN:
    raise ValueError('Set MODEL_FROZEN=True only after checkpoint and hyperparameters are frozen.')

experiment_config = {
    'model_name': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'preprocessing_version': PREPROCESSING_VERSION,
    'max_length': MAX_LENGTH,
    'seed': SEED,
    'labels': list(LABELS),
    'training_split': 'leakage_safe_train',
    'validation_split': 'official_validation',
    'test_split': 'official_test_only_after_selection',
    'igar_role': 'external_domain_validation_only',
    'model_frozen': MODEL_FROZEN,
    'test_evaluation_enabled': RUN_TEST_EVALUATION,
}
print(json.dumps(experiment_config, indent=2))

In [ ]:
if not RUN_TRAINING:
    print('Dry run: set RUN_TRAINING=True to download dependencies/checkpoint and fine-tune IndoBERT.')
else:
    from datasets import Dataset, DatasetDict

    def to_dataset(rows: list[PreparedRow]) -> Dataset:
        return Dataset.from_dict(
            {
                'text': [row.text for row in rows],
                'label': [row.label for row in rows],
                'label_id': [LABEL_TO_ID[row.label] for row in rows],
                'source_row_number': [row.source_row_number for row in rows],
            }
        )

    hf_dataset = DatasetDict(
        {
            'train': to_dataset(prepared_splits['train']),
            'validation': to_dataset(prepared_splits['validation']),
            'test': to_dataset(prepared_splits['test']),
        }
    )
    print(hf_dataset)

## 4. Optional IndoBERT fine-tuning and evaluation

This cell is skipped in the default dry run. It reports Accuracy, macro Precision, macro Recall, Macro F1, per-class metrics, and a confusion matrix.

In [ ]:
if RUN_TRAINING:
    import matplotlib.pyplot as plt
    import numpy as np
    from sklearn.metrics import (
        ConfusionMatrixDisplay,
        accuracy_score,
        classification_report,
        confusion_matrix,
        precision_recall_fscore_support,
    )
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        Trainer,
        TrainingArguments,
        set_seed,
    )

    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        num_labels=len(LABELS),
        id2label={index: label for label, index in LABEL_TO_ID.items()},
        label2id=LABEL_TO_ID,
    )

    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

    tokenized = hf_dataset.map(tokenize, batched=True)
    tokenized = tokenized.rename_column('label_id', 'labels')
    tokenized = tokenized.remove_columns(['text', 'label', 'source_row_number'])
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        if isinstance(logits, tuple):
            logits = logits[0]
        predictions = np.argmax(logits, axis=-1)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='macro', zero_division=0
        )
        return {
            'accuracy': accuracy_score(labels, predictions),
            'precision_macro': precision,
            'recall_macro': recall,
            'macro_f1': f1,
        }

    common_training_kwargs = dict(
        output_dir=str(ARTIFACT_DIR / 'indobert-checkpoint'),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        save_strategy='no',
        load_best_model_at_end=False,
        report_to=[],
        seed=SEED,
    )
    try:
        training_args = TrainingArguments(**common_training_kwargs, eval_strategy='epoch')
    except TypeError:
        training_args = TrainingArguments(
            **common_training_kwargs, evaluation_strategy='epoch'
        )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['validation'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    validation_metrics = trainer.evaluate(tokenized['validation'])
    metrics = {
        'config': experiment_config,
        'validation': validation_metrics,
    }
    if RUN_TEST_EVALUATION:
        test_output = trainer.predict(tokenized['test'])
        test_predictions = np.argmax(test_output.predictions, axis=-1)
        test_labels = test_output.label_ids
        per_class = classification_report(
            test_labels,
            test_predictions,
            labels=list(range(len(LABELS))),
            target_names=list(LABELS),
            output_dict=True,
            zero_division=0,
        )
        metrics['test'] = {
            'accuracy': accuracy_score(test_labels, test_predictions),
            'per_class': per_class,
            'confusion_matrix': confusion_matrix(
                test_labels, test_predictions, labels=list(range(len(LABELS)))
            ).tolist(),
        }
    else:
        print('Test evaluation skipped: freeze the model before enabling the confirmation-set gate.')
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    (ARTIFACT_DIR / 'indobert_metrics.json').write_text(
        json.dumps(metrics, indent=2, default=float) + '\n', encoding='utf-8'
    )
    if RUN_TEST_EVALUATION:
        ConfusionMatrixDisplay.from_predictions(
            test_labels, test_predictions, display_labels=LABELS, cmap='Blues'
        )
        plt.tight_layout()
        plt.savefig(ARTIFACT_DIR / 'indobert_confusion_matrix.png', dpi=160)
        plt.show()
    print(json.dumps(metrics, indent=2, default=float))

## 5. IGAR external/domain-validation sample

The acquisition script tracks a small, label-stratified sample of IGAR. Its labels are derived from app ratings and are noisy proxies; they are reported separately and must not influence training or hyperparameter selection.

In [ ]:
if IGAR_SAMPLE.exists():
    with IGAR_SAMPLE.open('r', encoding='utf-8-sig', newline='') as handle:
        igar_rows = list(csv.DictReader(handle))
    print('IGAR sample rows:', len(igar_rows))
    print('IGAR source labels:', dict(Counter(row['labelScoreBase'] for row in igar_rows)))
    print('Use only for external/domain validation after the SmSA baseline is frozen.')
else:
    print('IGAR sample is missing; run scripts/download_week2_datasets.py first.')